In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_ollama import OllamaEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

C:\Users\UseR\AppData\Local\Temp\ipykernel_25208\2916260490.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [4]:
len(docs)

9

In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_docs = splitter.split_documents(docs)

In [6]:
len(splitted_docs)

26

In [7]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vector_store = InMemoryVectorStore.from_documents(
    documents = splitted_docs,
    embedding=embeddings,

)

In [8]:
@tool
def ret_tool(query:str):
    """
        This tool can help retrieve data from thee PDF docs and these docs have details of medical reports.
    """

    print("tool called: ", query)

    docs = vector_store.similarity_search(query=query,  k=4)
    
    context = ""

    for doc in docs:
        context = doc.page_content + "\n"
    
    return context

In [9]:
ret_tool.invoke("Patient Name")

tool called:  Patient Name


'Consultant- Hemato-Oncopathology & \nFlowcytometry                                     \nNRL - Dr Lal PathLabs Ltd\nDr Sunanda\nMD, Pathology\nSr. Consultant Pathologist - \nHematology & Immunology                              \nNRL - Dr Lal PathLabs Ltd\n-------------------------------End of report --------------------------------\nAHEEEHAPMKHJBKHNLKIHBLLCBILLJCECCJLCIKPLPKEDFBFAPPAHEEEHA\nBNFFFNBPAPBOACIGFGELNGAPAOAHFHAKAKNOBCJKBLEKMCJGNPBNFFFNB\nCIEGCAFJLPNBKNNOPFIAAJHFJGDEHEDFLKPHENFLMLKIOEFLBKHDEHANP\nDJECMEFNBMHIEDINMIMELFMFAFPLBLAFJIFFOAPAKCLJPDNLIJFBKEMEK\nKDCILHFJONFFDIAJMBICKPJDCENJKPCHKKMBACOIPLKGKPNGKFFFOIOND\nFGCBBEFNNICGDKHKILIGCIPBADHFOFAINCFCBKDKBLIKOMNALNEJOGEKD\nDPIMIJFNELODDJHIEKMPCGFHIHAFJBALPPMPBLMIKJCHKJNKIJNFMDILD\nNJJJAMFMAKNKPELJGANHDOMILLBMFMBHIFCCAMNNIIKNOPNKBNFMBHILL\nNKIHJAFMDKOGFAAMFNOIHMIKHEJFEKFBJFFKBKOFOOBIOCNKDJPHNEKLJ\nNMDBPJFCEHNJJDCLCDBFGPNFIDFKPNJHKEPLELPLNKCKIGEOCIGKHMEJP\nICIKOLFLGKCBMMCMPNBDAJEEINMEFCBBJMFEBFOEONDMKPFKEPCIGKAMG\n'

In [10]:
llm = ChatGroq(model="openai/gpt-oss-20b")

In [11]:
System_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
    ALWAYS use the `ret_tool` tool for questions requiring external knowledge.
"""

In [12]:
agent = create_agent(
    model = llm,
    tools = [ret_tool],
    system_prompt = System_Prompt
)

In [17]:
query = "What is the name of patient? what is the name of doctors?"
response = agent.invoke({"messages":[{"role":"user", "content":query}]})

tool called:  patient name doctor name medical report


In [18]:
result = response["messages"][-1].content

In [19]:
print(result)

- **Patient name:** Ms. Nikita Chudhary  
- **Doctor’s name:** Dr. Nitin Nahar
